## Data Preprocessing
Multi-label tokenization, DataLoader construction, and saving to disk.
The original `labels` column (list of active emotion indices) is preserved as-is; multi-hot encoding happens at training time inside `EmotionDataset`.

In [1]:
import torch
from datasets import load_dataset
from transformers import BertTokenizer
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os

print("torch:", torch.__version__)

torch: 2.11.0+cpu


### 1. Load Dataset

In [2]:
dataset = load_dataset("go_emotions", "simplified")
label_names = dataset['train'].features['labels'].feature.names
print(dataset)
print(f"\n{len(label_names)} emotion classes:", label_names)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

28 emotion classes: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


### 2. Verify Multi-Label Structure
GoEmotions is natively multi-label — many sentences carry more than one emotion. We keep the full `labels` list rather than collapsing to a single class.

In [ ]:
# Show a few examples with multiple labels to confirm the structure
multi_label_examples = [ex for ex in dataset['train'] if len(ex['labels']) > 1][:5]
for ex in multi_label_examples:
    emotion_str = ', '.join(label_names[l] for l in ex['labels'])
    print(f"labels: [{emotion_str}]")
    print(f"text:   {ex['text'][:80]}")
    print()

### 3. Tokenize
Use `bert-base-uncased` tokenizer with padding and truncation to max 128 tokens.

In [ ]:
tokenizer = BertTokenizer.from_pretrained("google-bert/bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

tokenized = dataset.map(tokenize, batched=True)
# 'labels' (variable-length list) is intentionally excluded — handled in EmotionDataset
tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids'])

print(tokenized)
print("\nSample input_ids shape:", tokenized['train'][0]['input_ids'].shape)

### 4. PyTorch Dataset & DataLoaders

In [ ]:
import torch

NUM_LABELS = len(label_names)

class EmotionDataset(Dataset):
    def __init__(self, hf_split, num_labels=NUM_LABELS):
        self.data       = hf_split
        self.num_labels = num_labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        label_vec = torch.zeros(self.num_labels, dtype=torch.float)
        for lbl in row['labels']:
            label_vec[lbl] = 1.0
        return {
            'input_ids':      row['input_ids'],
            'attention_mask': row['attention_mask'],
            'token_type_ids': row['token_type_ids'],
            'labels':         label_vec,
        }

train_dataset = EmotionDataset(tokenized['train'])
val_dataset   = EmotionDataset(tokenized['validation'])
test_dataset  = EmotionDataset(tokenized['test'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
# Verify one batch — labels should be [32, 28] float multi-hot vectors
batch = next(iter(train_loader))
for k, v in batch.items():
    print(f"{k}: {v.shape}  dtype={v.dtype}")

# Show the multi-hot vector for the first sample
active = batch['labels'][0].nonzero(as_tuple=True)[0].tolist()
print(f"\nFirst sample active labels: {[label_names[i] for i in active]}")

### 5. Save to Disk

In [7]:
os.makedirs("processed", exist_ok=True)

tokenized.save_to_disk("processed/go_emotions_tokenized")
print("Saved to processed/go_emotions_tokenized")

Saving the dataset (0/1 shards):   0%|          | 0/43410 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5426 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5427 [00:00<?, ? examples/s]

Saved to processed/go_emotions_tokenized


In [ ]:
# Verify reload — confirm 'labels' column is preserved
from datasets import load_from_disk

reloaded = load_from_disk("processed/go_emotions_tokenized")
reloaded.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids'])
print("Reloaded:", reloaded)

ex = reloaded['train'][0]
print(f"\nFirst train labels (raw list): {ex['labels']}")
print(f"Emotions: {[label_names[l] for l in ex['labels']]}")